# SGDR — Stochastic Gradient Descent with Warm Restarts (2016)

_Papers · Foundations & Optimization_

**Decay the learning rate along a cosine curve, then jump it back up and restart — repeatedly — to train faster and collect a free ensemble of snapshots.**

---

This notebook is a practice scaffold for the **SGDR — Stochastic Gradient Descent with Warm Restarts (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import math, torch

# ---- SGDR LR SCHEDULE FROM SCRATCH (Eq. 5, Loshchilov & Hutter 2016) ----
class SGDR:
    """Cosine annealing with warm restarts. Returns the LR for each call to .get();
    advance the T_cur clock with .step(). eta_max is the optimizer's base lr."""
    def __init__(self, eta_max, eta_min=0.0, T_0=4, T_mult=1):
        self.eta_max, self.eta_min = eta_max, eta_min
        self.T_mult = T_mult
        self.T_cur, self.T_i = 0, T_0
    def get(self):
        # Eq. (5): eta = eta_min + 0.5*(eta_max-eta_min)*(1 + cos(pi * T_cur / T_i))
        return self.eta_min + 0.5*(self.eta_max-self.eta_min)*(1 + math.cos(math.pi*self.T_cur/self.T_i))
    def step(self):
        self.T_cur += 1
        if self.T_cur >= self.T_i:        # restart: snap T_cur back to 0, grow the run
            self.T_cur = 0
            self.T_i *= self.T_mult

# ---- recompute the WORKED EXAMPLE: eta_max=0.1, eta_min=0, T_0=4, T_mult=1 ----
s = SGDR(eta_max=0.1, eta_min=0.0, T_0=4, T_mult=1)
seq = []
for _ in range(5):            # 4 steps of run 1, then the restart back to 0.1
    seq.append(round(s.get(), 5)); s.step()
print("worked example LRs:", seq)   # [0.1, 0.08536, 0.05, 0.01464, 0.1]
assert seq == [0.1, 0.08536, 0.05, 0.01464, 0.1]

# ---- THE ORACLE: my schedule must equal torch's CosineAnnealingWarmRestarts ----
# A toy 1-param SGD optimizer; we only read its learning rate each step.
ETA_MAX, ETA_MIN, T0, TMULT, STEPS = 0.1, 0.0, 3, 2, 12   # crosses 2 restarts
p = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([p], lr=ETA_MAX)
sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt, T_0=T0, T_mult=TMULT, eta_min=ETA_MIN)

mine = SGDR(eta_max=ETA_MAX, eta_min=ETA_MIN, T_0=T0, T_mult=TMULT)
torch_lrs, my_lrs = [], []
for _ in range(STEPS):
    torch_lrs.append(opt.param_groups[0]["lr"])  # torch's current lr
    my_lrs.append(mine.get())                    # my current lr
    sched.step(); mine.step()                    # advance both clocks together

t = torch.tensor(torch_lrs); m = torch.tensor(my_lrs)
print("torch lrs:", [round(x,5) for x in torch_lrs])
print("my lrs   :", [round(x,5) for x in my_lrs])
print("allclose (atol=1e-6):", torch.allclose(m, t, atol=1e-6))   # expect True

# ---- use it: one SGD epoch where the lr follows the schedule ----
w = torch.tensor([5.0])                  # minimize (w-1)^2, gradient = 2(w-1)
sgdr = SGDR(eta_max=0.3, eta_min=0.0, T_0=5, T_mult=1)
for _ in range(20):
    g = 2*(w - 1.0)
    w = w - sgdr.get()*g                 # SGD step at the scheduled lr
    sgdr.step()
print("converged w (target 1.0):", round(w.item(), 4))

## Visualize the data & results

_What does the SGDR cosine-with-warm-restarts learning rate look like over time, and does it actually help on a non-convex toy problem — can the restart kick the optimizer OUT of a shallow local minimum and into the deeper global one, the way the paper's intuition suggests?_

In [ ]:
import math

def sgdr_lr(step, eta_max=0.5, eta_min=0.0, T_0=10, T_mult=2):
    T_cur, T_i = step, T_0
    while T_cur >= T_i:                 # unwind to the current run
        T_cur -= T_i; T_i *= T_mult
    return eta_min + 0.5*(eta_max-eta_min)*(1 + math.cos(math.pi*T_cur/T_i))

# LEFT chart: the schedule itself
print("schedule:", [round(sgdr_lr(s),4) for s in range(0,71)])

# RIGHT chart: escape a shallow local minimum on a NON-CONVEX double well.
# f(x)=0.5*(x^2-1)^2 + 0.6x : shallow local min x=+0.79 (f=+0.54), deep global min x=-1.13 (f=-0.64).
import random
def f(x):  return 0.5*(x*x-1)**2 + 0.6*x
def gp(x): return 2*x*(x*x-1) + 0.6           # df/dx

def run(get_lr, seed, steps=80, noise=0.9):
    rng = random.Random(seed); x = 0.6        # start INSIDE the shallow basin
    losses = []
    for s in range(steps):
        n = rng.uniform(-noise, noise)         # stochastic (mini-batch) gradient noise
        x = x - get_lr(s)*(gp(x) + n)          # noisy SGD step at the scheduled lr
        losses.append(f(x))
    return losses, x

seeds = range(1, 13)
def avg(get_lr):
    runs = [run(get_lr, s)[0] for s in seeds]
    return [round(sum(r[i] for r in runs)/len(runs), 4) for i in range(80)]

const = avg(lambda s: 0.05)
sgdr  = avg(lambda s: sgdr_lr(s, eta_max=0.45))
esc_const = sum(run(lambda s: 0.05, s)[1] < 0 for s in seeds)
esc_sgdr  = sum(run(lambda s: sgdr_lr(s, eta_max=0.45), s)[1] < 0 for s in seeds)
print("constant final avg loss:", const[-1], " runs that escaped to global min:", esc_const, "/12")
print("SGDR     final avg loss:", sgdr[-1],  " runs that escaped to global min:", esc_sgdr,  "/12")
print("Bump at first restart (step 9 -> 10):", sgdr[9], "->", sgdr[10])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Using Eq. (5) with $\eta_{max}=0.2$, $\eta_{min}=0$, $T_0=2$, $T_{mult}=1$, what is the learning rate at $T_{cur}=0,\ 1,\ 2$? (Note $T_{cur}=2$ triggers a restart.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $T_{cur}=0$: $\cos(0)=1$, so $\eta=0.1(1+1)=0.2$. — _Start of a run is always $\eta_{max}$._
- $T_{cur}=1$: $\cos(\pi\cdot 1/2)=\cos(\pi/2)=0$, so $\eta=0.1(1+0)=0.1$. — _Halfway through a 2-epoch run the learning rate is the midpoint._
- $T_{cur}=2$ would give $\cos(\pi)=-1\Rightarrow\eta=0$, but this is a restart: $T_{cur}$ resets to $0$, so $\eta$ jumps back to $0.2$. — _The warm restart snaps the learning rate back to $\eta_{max}$._

**Answer:** $0.2,\ 0.1$, then a jump back to $0.2$. The schedule never lingers at $0$ &mdash; it bottoms out and immediately restarts.

</details>

**Problem 2.** ABLATION: train a tiny model with a CONSTANT learning rate vs the SGDR cosine-with-restarts schedule for the same number of steps. What do you expect for (a) the final loss and (b) the shape of the SGDR loss curve at each restart?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run both schedules from the same initialization and seed (see CODE / CODEVIZ). — _Only the learning-rate-vs-time curve differs, so any gap is due to the schedule._
- Compare final losses. — _SGDR's low-learning-rate phase at the bottom of each cycle lets it settle tightly into a minimum, which constant lr cannot do._
- Look at the SGDR loss right after each restart. — _The learning rate jumps to $\eta_{max}$, so the weights are kicked and the loss briefly RISES, then re-descends._

**Answer:** SGDR ends at a lower loss than a constant learning rate (the cosine floor lets it converge tightly), and its loss curve shows a small bump up at every restart followed by a re-descent &mdash; the visible signature of escaping the current minimum. These are OUR small-run numbers (CODEVIZ), not the paper's reported metrics.

</details>

**Problem 3.** You set $T_0=10$, $T_{mult}=2$. How long are the first three runs, and at which global epochs do the restarts occur?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run 1 length $=T_0=10$ (epochs 0&ndash;9), restart at epoch 10. — _The first run uses $T_0$._
- Run 2 length $=T_{mult}\cdot 10=20$ (epochs 10&ndash;29), restart at epoch 30. — _Each run length is multiplied by $T_{mult}=2$._
- Run 3 length $=2\cdot 20=40$ (epochs 30&ndash;69), restart at epoch 70. — _Lengths grow geometrically: $10,20,40,\dots$._

**Answer:** Run lengths $10,\ 20,\ 40$; restarts at global epochs $10,\ 30,\ 70$. Doubling makes early restarts frequent (explore) and later runs long (fine convergence).

</details>